#### Imports

In [0]:
from pyspark.sql.functions import *


#### Table Configuration

In [0]:
# Table configuration

bronze_table = (
    "workspace.nyc_taxi_aws.bronze_taxi_trips"
)

silver_table = (
    "workspace.nyc_taxi_aws.silver_taxi_trips_curated"
)

silver_checkpoint = (
    "s3://ashish-nyc-taxi-lakehouse/checkpoints/silver"
)

print("===================================")
print("SILVER NOTEBOOK STARTED")
print("===================================")

print(
    f"Bronze table: {bronze_table}"
)

print(
    f"Silver table: {silver_table}"
)

print(
    f"Silver checkpoint: {silver_checkpoint}"
)

SILVER NOTEBOOK STARTED
Bronze table: workspace.nyc_taxi_aws.bronze_taxi_trips
Silver table: workspace.nyc_taxi_aws.silver_taxi_trips_curated
Silver checkpoint: s3://ashish-nyc-taxi-lakehouse/checkpoints/silver


#### Creating Bronze DataFrame from bronze table

In [0]:
bronze_df = (
    spark.readStream
    .table(bronze_table)
)

#### Creating Silver DataFrame using bronze df 

In [0]:
silver_df = (
    bronze_df
    .withColumnRenamed( #Changed the column names to more clear names
        "tpep_pickup_datetime",
        "pickup_datetime"
    )
    .withColumnRenamed(
        "tpep_dropoff_datetime",
        "dropoff_datetime"
    )
)

#### Updating Silver DataFrame and adding new columns

In [0]:
silver_df = (
    silver_df
    .withColumn(
        "pickup_datetime",
        to_timestamp("pickup_datetime")
    )
    .withColumn(
        "dropoff_datetime",
        to_timestamp("dropoff_datetime")
    )
    .withColumn(
        "trip_distance",
        col("trip_distance").cast("double")
    )
    .withColumn(
        "fare_amount",
        col("fare_amount").cast("double")
    )
    .withColumn(
        "total_amount",
        col("total_amount").cast("double")
    )
    .withColumn(
        "passenger_count",
        col("passenger_count").cast("integer")
    )
    .withColumn(
        "payment_type",
        col("payment_type").cast("integer")
    )
)

#### Updating the new columns

In [0]:
silver_df = (
    silver_df
    .withColumn(
        "trip_duration_minutes",
        round(
            (
                unix_timestamp("dropoff_datetime")
                - unix_timestamp("pickup_datetime")
            ) / 60,
            2
        )
    )
    .withColumn(
        "pickup_date",
        to_date("pickup_datetime")
    )
    .withColumn(
        "pickup_hour",
        hour("pickup_datetime")
    )
    .withColumn(
        "pickup_month",
        month("pickup_datetime")
    )
    .withColumn(
        "pickup_year",
        year("pickup_datetime")
    )
)

#### Conditioning the data to remove invalid data

In [0]:
valid_condition = (
    col("pickup_datetime").isNotNull()
    & col("dropoff_datetime").isNotNull()
    & (
        col("dropoff_datetime")
        > col("pickup_datetime")
    )
    & (
        col("trip_distance") > 0
    )
    & (
        col("fare_amount") >= 0
    )
    & (
        col("total_amount") >= 0
    )
    & (
        col("trip_duration_minutes") > 0
    )
    & (
        col("trip_duration_minutes") <= 1440
    )
)

clean_silver_df = (
    silver_df
    .filter(valid_condition)
)

#### Added Hash column to identify duplicates

In [0]:
clean_silver_df = (
    clean_silver_df
    .withColumn(
        "trip_record_hash",
        sha2(
            concat_ws(
                "||",
                col("VendorID").cast("string"),
                col("pickup_datetime").cast("string"),
                col("dropoff_datetime").cast("string"),
                col("PULocationID").cast("string"),
                col("DOLocationID").cast("string"),
                col("passenger_count").cast("string"),
                col("trip_distance").cast("string"),
                col("fare_amount").cast("string"),
                col("extra").cast("string"),
                col("mta_tax").cast("string"),
                col("tip_amount").cast("string"),
                col("tolls_amount").cast("string"),
                col("total_amount").cast("string"),
                col("payment_type").cast("string"),
                col("RatecodeID").cast("string"),
                col("store_and_fwd_flag").cast("string")
            ),
            256
        )
    )
)

#### Dropping duplicates from cleaned data

In [0]:
clean_silver_df = (
    clean_silver_df
    .dropDuplicates(
        [
            "trip_record_hash"
        ]
    )
)

#### Adding new columns for aggregation purpose

In [0]:
clean_silver_df = (
    clean_silver_df
    .withColumn(
        "fare_per_mile",
        round(
            col("fare_amount")
            / col("trip_distance"),
            2
        )
    )
    .withColumn(
        "average_speed_mph",
        round(
            col("trip_distance")
            / (
                col("trip_duration_minutes")
                / 60
            ),
            2
        )
    )
)

#### Creating final Silver delta Table

In [0]:
silver_query = (
    clean_silver_df.writeStream
    .format("delta")
    .outputMode("append")
    .option(
        "checkpointLocation",
        silver_checkpoint
    )
    .trigger(
        availableNow=True
    )
    .toTable(
        silver_table
    )
)

#### Terminate the stream

In [0]:
silver_query.awaitTermination()

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-4660997542170792>, line 1
----> 1 silver_query.awaitTermination()

NameError: name 'silver_query' is not defined

#### Updated log for incremental file

In [0]:
print("===================================")
print("SILVER STREAM PROGRESS")
print("===================================")

print(
    silver_query.lastProgress
)

SILVER STREAM PROGRESS


---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-4660997542170794>, line 6
      2 print("SILVER STREAM PROGRESS")
      3 print("===================================")
      5 print(
----> 6     silver_query.lastProgress
      7 )

NameError: name 'silver_query' is not defined

#### Exception

In [0]:
if silver_query.exception() is not None:

    raise Exception(
        silver_query.exception()
    )

else:

    print(
        "Silver streaming completed successfully."
    )

#### Count added to the delta table

In [0]:
silver_count = (
    spark.table(
        silver_table
    ).count()
)

print(
    f"Silver record count: {silver_count}"
)